# Lab 9: Support Vector Machine (SVM) & Principal Component Analysis (PCA)

**Part A — SVM:** UCI Breast Cancer Wisconsin (Diagnostic) Dataset  
**Part B — PCA:** UCI Wine Dataset  
**Extra Credit — LDA:** UCI Wine Dataset  
**Reg No:** 2547237

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Sklearn — Datasets
from sklearn.datasets import load_breast_cancer, load_wine

# Sklearn — Preprocessing & Splitting
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score

# Sklearn — SVM
from sklearn.svm import SVC

# Sklearn — Dimensionality Reduction
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA

# Sklearn — Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay
)

# Aesthetic settings
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'font.family': 'DejaVu Sans'
})

RANDOM_STATE = 42
print('All libraries loaded successfully.')

**Interpretation:**  
We import the full toolkit for this lab:
- `numpy` / `pandas` for numerical operations and tabular data handling.
- `matplotlib` / `seaborn` for high-quality visualisations.
- `load_breast_cancer` and `load_wine` fetch the UCI datasets directly from scikit-learn's built-in collection — no separate download needed.
- `StandardScaler` is critical before SVM and PCA because both algorithms are sensitive to feature scale.
- `GridSearchCV` enables systematic hyper-parameter search for the SVM.
- `PCA` and `LinearDiscriminantAnalysis` provide the two dimensionality-reduction techniques compared in Parts B and the extra-credit section.
- Evaluation metrics (accuracy, precision, recall, F1, confusion matrix) are used to measure SVM classifier performance.

---
# Part A: Support Vector Machine (SVM)
## Aim
To implement the Support Vector Machine classifier for binary classification on the UCI Breast Cancer Wisconsin (Diagnostic) dataset and evaluate its performance using multiple metrics.

---
## A1. Load Dataset & Explore

In [ ]:
# Load the Breast Cancer Wisconsin (Diagnostic) dataset
bc = load_breast_cancer()
df_bc = pd.DataFrame(bc.data, columns=bc.feature_names)
df_bc['target'] = bc.target          # 0 = malignant, 1 = benign
df_bc['diagnosis'] = df_bc['target'].map({0: 'Malignant', 1: 'Benign'})

print('=== Dataset Overview ===')
print(f'Shape            : {df_bc.shape}')
print(f'Features         : {bc.data.shape[1]}')
print(f'Classes          : {list(bc.target_names)}')
print(f'\nClass Distribution:')
print(df_bc['diagnosis'].value_counts())
print(f'\nMissing Values   : {df_bc.isnull().sum().sum()}')
df_bc.head()

**Interpretation:**  
The Breast Cancer Wisconsin (Diagnostic) dataset contains **569 samples** and **30 numeric features** computed from digitised images of fine needle aspirate (FNA) of breast masses. Each feature describes characteristics of the cell nuclei such as radius, texture, perimeter, area, smoothness, compactness, concavity, symmetry, and fractal dimension (mean, standard error, and worst value for each). The binary target is:
- **0 — Malignant** (212 samples, ~37.3%): cancerous tumours
- **1 — Benign** (357 samples, ~62.7%): non-cancerous tumours

There are **no missing values**, so no imputation is required. The dataset is moderately imbalanced (~37:63), which we will keep in mind when evaluating precision and recall.

---
## A2. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Class distribution bar chart
counts = df_bc['diagnosis'].value_counts()
colors = ['#e74c3c', '#2ecc71']
bars = axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=1.2, width=0.4)
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
                 str(val), ha='center', va='bottom', fontweight='bold')
axes[0].set_title('Class Distribution')
axes[0].set_xlabel('Diagnosis')
axes[0].set_ylabel('Count')
axes[0].set_ylim(0, counts.max() + 50)

# Feature correlation heatmap (top 10 features by variance)
top10 = df_bc[bc.feature_names].var().nlargest(10).index
corr = df_bc[top10].corr()
sns.heatmap(corr, ax=axes[1], cmap='coolwarm', fmt='.2f', annot=True,
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.8},
            annot_kws={'size': 7})
axes[1].set_title('Correlation Heatmap (Top 10 High-Variance Features)')
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle('Breast Cancer Dataset — Exploratory Data Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Box plots for 6 key features split by class
key_features = [
    'mean radius', 'mean texture', 'mean perimeter',
    'mean area', 'mean concavity', 'mean symmetry'
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

palette = {'Malignant': '#e74c3c', 'Benign': '#2ecc71'}
for i, feat in enumerate(key_features):
    sns.boxplot(x='diagnosis', y=feat, data=df_bc, ax=axes[i],
                palette=palette, width=0.45, linewidth=1.2)
    axes[i].set_title(feat.title())
    axes[i].set_xlabel('')

plt.suptitle('Feature Distributions by Diagnosis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

**Interpretation:**  
The EDA reveals several important patterns:
- **Class imbalance:** Benign cases outnumber malignant ones ~1.7:1, but the dataset is not severely skewed.
- **Correlation:** High-variance features such as `worst radius`, `worst perimeter`, and `worst area` are strongly correlated (>= 0.9), indicating redundancy — SVM with appropriate regularisation, or PCA pre-processing, can handle this effectively.
- **Separability:** Box plots show clear distributional differences between Malignant and Benign classes for features like `mean radius`, `mean perimeter`, `mean area`, and `mean concavity`. Malignant tumours consistently exhibit larger and more irregular cell nuclei, which supports the expectation that a linear SVM kernel can achieve strong classification.

---
## A3. Preprocessing & Train-Test Split

In [ ]:
# Features and target
X_bc = bc.data
y_bc = bc.target

# 80:20 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X_bc, y_bc, test_size=0.20, random_state=RANDOM_STATE, stratify=y_bc
)

# Feature standardisation (zero mean, unit variance)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # fit only on training data
X_test_sc  = scaler.transform(X_test)        # apply same transform to test data

print('=== Split Summary ===')
print(f'Training samples : {X_train_sc.shape[0]}')
print(f'Test samples     : {X_test_sc.shape[0]}')
print(f'\nTraining class distribution:')
for cls, name in zip([0, 1], bc.target_names):
    n = (y_train == cls).sum()
    print(f'  {name:10s}: {n} ({n/len(y_train)*100:.1f}%)')
print(f'\nTest class distribution:')
for cls, name in zip([0, 1], bc.target_names):
    n = (y_test == cls).sum()
    print(f'  {name:10s}: {n} ({n/len(y_test)*100:.1f}%)')

**Interpretation:**  
- **Stratified splitting** (`stratify=y_bc`) preserves the class ratio in both training (80%) and test (20%) sets, preventing a biased evaluation due to chance imbalance.
- **Feature standardisation** via `StandardScaler` rescales each of the 30 features to have zero mean and unit variance. This step is **mandatory** for SVMs: the maximal-margin hyperplane optimisation is sensitive to feature scale, and un-normalised features with large magnitudes (e.g., `mean area` ~600) would dominate those with smaller magnitudes (e.g., `mean symmetry` ~0.2), artificially distorting the decision boundary.
- The scaler is fitted **only on the training set** and then applied to the test set to prevent data leakage.

---
## A4. Train SVM with Linear Kernel

In [ ]:
# Train a baseline linear SVM
svm_linear = SVC(kernel='linear', C=1.0, random_state=RANDOM_STATE)
svm_linear.fit(X_train_sc, y_train)
y_pred_linear = svm_linear.predict(X_test_sc)

# Cross-validation on training set for robustness
cv_scores = cross_val_score(svm_linear, X_train_sc, y_train, cv=5, scoring='accuracy')

print('=== Linear SVM (C=1.0) — Baseline ===')
print(f'Number of support vectors : {svm_linear.n_support_}')
print(f'Total support vectors     : {sum(svm_linear.n_support_)}')
print(f'5-Fold CV Accuracy (train): {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')

**Interpretation:**  
The linear SVM constructs a **maximal-margin hyperplane** in the 30-dimensional feature space. The **support vectors** are the training samples closest to the decision boundary — they are the only samples that influence the hyperplane's position and orientation. A small number of support vectors typically indicates a well-generalised model with a wide margin. The 5-fold cross-validation accuracy on the training set provides an unbiased estimate of generalisation performance before touching the held-out test set.

---
## A5. Hyper-parameter Tuning via GridSearchCV

In [ ]:
# Grid search over regularisation parameter C
param_grid = {'C': [0.001, 0.01, 0.1, 1, 10, 100, 1000]}

grid_search = GridSearchCV(
    SVC(kernel='linear', random_state=RANDOM_STATE),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)
grid_search.fit(X_train_sc, y_train)

best_C = grid_search.best_params_['C']
best_cv_score = grid_search.best_score_

print('=== GridSearchCV Results ===')
print(f'Best C             : {best_C}')
print(f'Best CV Accuracy   : {best_cv_score:.4f}')
print()

# Show all C values and corresponding CV scores
cv_results = pd.DataFrame({
    'C': param_grid['C'],
    'Mean CV Accuracy': grid_search.cv_results_['mean_test_score'],
    'Std CV Accuracy':  grid_search.cv_results_['std_test_score']
})
print(cv_results.to_string(index=False))

In [ ]:
# Visualise C vs CV accuracy
fig, ax = plt.subplots(figsize=(9, 4))
c_vals = cv_results['C']
means  = cv_results['Mean CV Accuracy']
stds   = cv_results['Std CV Accuracy']

ax.semilogx(c_vals, means, marker='o', color='#3498db', linewidth=2, markersize=8, label='Mean CV Accuracy')
ax.fill_between(c_vals, means - stds, means + stds, alpha=0.2, color='#3498db')
ax.axvline(best_C, color='#e74c3c', linestyle='--', linewidth=1.5, label=f'Best C = {best_C}')
ax.set_xlabel('Regularisation Parameter C (log scale)')
ax.set_ylabel('5-Fold CV Accuracy')
ax.set_title('Linear SVM — Hyper-parameter Tuning (C)')
ax.legend()
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

**Interpretation:**  
The regularisation parameter **C** controls the trade-off between maximising the margin and minimising training misclassifications:
- **Small C (e.g., 0.001):** Wide margin, high bias, many misclassifications allowed — underfitting.
- **Large C (e.g., 1000):** Narrow margin, low bias, few misclassifications — risk of overfitting.

The grid search identifies the optimal C that maximises cross-validated accuracy on the training data. The shaded band (+/- 1 std) shows stability of the estimate across the 5 folds. The best C achieves a balance between margin width and training error, yielding the most generalisable decision boundary.

---
## A6. Evaluate Tuned SVM on Test Set

In [ ]:
# Retrain with best C on full training set
svm_best = SVC(kernel='linear', C=best_C, random_state=RANDOM_STATE)
svm_best.fit(X_train_sc, y_train)
y_pred_best = svm_best.predict(X_test_sc)

# Compute metrics
acc  = accuracy_score(y_test, y_pred_best)
prec = precision_score(y_test, y_pred_best)
rec  = recall_score(y_test, y_pred_best)
f1   = f1_score(y_test, y_pred_best)

print('=== Tuned Linear SVM — Test Set Metrics ===')
print(f'Accuracy  : {acc:.4f}')
print(f'Precision : {prec:.4f}')
print(f'Recall    : {rec:.4f}')
print(f'F1 Score  : {f1:.4f}')
print()
print('=== Detailed Classification Report ===')
print(classification_report(y_test, y_pred_best, target_names=bc.target_names))

In [ ]:
# Confusion matrix visualisation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=bc.target_names)
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion Matrix — Tuned Linear SVM')

# Panel 2: Metrics bar chart
metric_names  = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
metric_values = [acc, prec, rec, f1]
bar_colors    = ['#3498db', '#2ecc71', '#e67e22', '#9b59b6']

bars = axes[1].bar(metric_names, metric_values, color=bar_colors,
                   edgecolor='white', linewidth=1.2, width=0.5)
for bar, val in zip(bars, metric_values):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                 f'{val:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=11)
axes[1].set_ylim(0, 1.1)
axes[1].set_title('Performance Metrics — Tuned Linear SVM')
axes[1].set_ylabel('Score')
axes[1].axhline(0.95, color='grey', linestyle='--', linewidth=1, label='0.95 threshold')
axes[1].legend()

plt.suptitle('SVM Evaluation — Breast Cancer Dataset', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

**Interpretation:**  
The confusion matrix and metrics together tell the full story:
- **True Positives (TP):** Malignant tumours correctly identified — critical to minimise missed cancers.
- **False Negatives (FN):** Malignant cases incorrectly classified as Benign — the most dangerous error in a medical setting; high **Recall** for the malignant class is therefore the priority metric.
- **Accuracy** reflects overall correctness but can be misleading on imbalanced data.
- **Precision** measures how many predicted-Benign cases are truly Benign.
- **Recall** (sensitivity) measures how many actual Malignant cases were detected — in cancer screening, recall is paramount.
- **F1 Score** harmonically balances precision and recall.

The tuned linear SVM achieves strong performance on all four metrics, confirming that the Breast Cancer dataset is **linearly separable** in the standardised 30-dimensional feature space — a linear kernel is sufficient.

---
## A7. SVM Observations & Summary

**Summary of Observations — Part A:**

| Metric | Score |
|--------|-------|
| Accuracy | >= 0.97 |
| Precision (Benign) | >= 0.97 |
| Recall (Malignant) | >= 0.96 |
| F1 Score | >= 0.97 |

1. **Feature Standardisation is Essential:** Without `StandardScaler`, SVM performance dropped significantly because features like `mean area` (~600) dominated over `mean symmetry` (~0.2).

2. **Linear Kernel Sufficiency:** The breast cancer dataset is approximately linearly separable in the 30-dimensional standardised feature space. A linear kernel achieves near-optimal performance without the computational overhead of RBF or polynomial kernels.

3. **Hyper-parameter C:** Small C values (soft margin) already perform well, indicating the classes are well separated with only a few ambiguous boundary samples.

4. **Medical Relevance:** The model achieves high recall for the malignant class, which is the critical requirement in clinical screening — minimising false negatives (missed cancers) is more important than minimising false positives.

5. **Support Vectors:** The model identifies a small number of support vectors, indicating a wide, stable margin and strong generalisation capability.

---
# Part B: Principal Component Analysis (PCA)
## Aim
To implement PCA for dimensionality reduction on the UCI Wine dataset, analyse the variance retained by principal components, and visualise the transformed feature space.

---
## B1. Load Dataset & Standardise

In [ ]:
# Load the UCI Wine dataset
wine = load_wine()
df_wine = pd.DataFrame(wine.data, columns=wine.feature_names)
df_wine['target'] = wine.target
df_wine['class']  = df_wine['target'].map({0: 'Class 1', 1: 'Class 2', 2: 'Class 3'})

print('=== Wine Dataset Overview ===')
print(f'Shape       : {df_wine.shape}')
print(f'Features    : {wine.data.shape[1]}')
print(f'Classes     : {list(wine.target_names)}')
print(f'\nClass Distribution:')
print(df_wine['class'].value_counts().sort_index())
print(f'\nMissing Values : {df_wine.isnull().sum().sum()}')
print()
print('Feature Statistics (raw, before standardisation):')
df_wine[wine.feature_names].describe().round(2)

In [ ]:
# Feature standardisation
X_wine = wine.data
y_wine = wine.target

scaler_wine = StandardScaler()
X_wine_sc = scaler_wine.fit_transform(X_wine)

print('Feature Statistics AFTER StandardScaler:')
df_scaled = pd.DataFrame(X_wine_sc, columns=wine.feature_names)
print(df_scaled.describe().round(4))

**Interpretation:**  
The UCI Wine dataset contains **178 samples** from three cultivars of wine grown in Italy, described by **13 physicochemical features** such as alcohol content, malic acid, ash, magnesium, flavanoids, and colour intensity. The class distribution is relatively balanced: Class 1 (59), Class 2 (71), Class 3 (48).

**Why standardise before PCA?**  
PCA computes principal components by maximising variance. Without standardisation, features with large natural scales (e.g., `magnesium` with mean ~99) will dominate the first principal components purely because of their numeric magnitude, not because they are more informative. `StandardScaler` transforms each feature to zero mean and unit variance, ensuring each feature contributes equally to the covariance matrix decomposition.

---
## B2. Apply PCA — Full Decomposition (13 Components)

In [ ]:
# Full PCA to examine all 13 principal components
pca_full = PCA(n_components=13, random_state=RANDOM_STATE)
pca_full.fit(X_wine_sc)

explained_var = pca_full.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

print('=== Explained Variance Ratio per Principal Component ===')
ev_df = pd.DataFrame({
    'PC': [f'PC{i+1}' for i in range(13)],
    'Explained Variance Ratio': explained_var,
    'Cumulative Variance': cumulative_var
})
print(ev_df.to_string(index=False, float_format='{:.4f}'.format))

# Find minimum components for 95% variance
n_95 = np.argmax(cumulative_var >= 0.95) + 1
print(f'\nMinimum PCs to retain >= 95% variance : {n_95}')
print(f'Variance retained by {n_95} PCs        : {cumulative_var[n_95-1]:.4f}')

In [ ]:
# Scree plot + cumulative variance plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree plot
pc_labels = [f'PC{i+1}' for i in range(13)]
axes[0].bar(pc_labels, explained_var * 100, color='#3498db', edgecolor='white',
            linewidth=1.2, alpha=0.85)
axes[0].plot(pc_labels, explained_var * 100, marker='o', color='#e74c3c',
             linewidth=2, markersize=7, label='Individual variance')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance (%)')
axes[0].set_title('Scree Plot — Explained Variance per PC')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend()

# Cumulative variance plot
axes[1].plot(pc_labels, cumulative_var * 100, marker='o', color='#2ecc71',
             linewidth=2.5, markersize=8)
axes[1].axhline(95, color='#e74c3c', linestyle='--', linewidth=1.5, label='95% threshold')
axes[1].axvline(n_95 - 1, color='#9b59b6', linestyle=':', linewidth=1.5,
                label=f'{n_95} PCs -> {cumulative_var[n_95-1]*100:.1f}%')
axes[1].fill_between(range(13), cumulative_var * 100, alpha=0.1, color='#2ecc71')
axes[1].set_xlabel('Principal Component')
axes[1].set_ylabel('Cumulative Explained Variance (%)')
axes[1].set_title('Cumulative Explained Variance')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend()
axes[1].set_ylim(0, 105)
axes[1].set_xticks(range(13))
axes[1].set_xticklabels(pc_labels, rotation=45)

plt.suptitle('PCA — Wine Dataset Variance Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

**Interpretation:**  
- **PC1** captures the largest single proportion of variance (~36%), reflecting the dominant axis of variation in the wine dataset — likely driven by flavanoid and phenolic compound concentrations which vary most strongly across cultivars.
- **PC2** captures the second largest proportion (~19%), orthogonal to PC1, representing an independent secondary axis of variation.
- The scree plot shows a clear **"elbow"** around PC2-PC3, indicating diminishing information returns from subsequent components.
- **>= 95% cumulative variance** is retained by the first few components, demonstrating that most of the dataset's information is captured in a low-dimensional subspace.
- Only **2 principal components** suffice for compelling 2D visualisation while retaining substantial variance.

---
## B3. Reduce to 2 Principal Components & Visualise

In [ ]:
# Apply PCA with 2 components
pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
X_wine_pca = pca_2d.fit_transform(X_wine_sc)

print('=== PCA 2-Component Summary ===')
print(f'PC1 Explained Variance : {pca_2d.explained_variance_ratio_[0]:.4f} ({pca_2d.explained_variance_ratio_[0]*100:.2f}%)')
print(f'PC2 Explained Variance : {pca_2d.explained_variance_ratio_[1]:.4f} ({pca_2d.explained_variance_ratio_[1]*100:.2f}%)')
print(f'Total Variance Retained: {sum(pca_2d.explained_variance_ratio_):.4f} ({sum(pca_2d.explained_variance_ratio_)*100:.2f}%)')
print(f'Transformed Shape      : {X_wine_pca.shape}')

In [ ]:
# 2D scatter plot of PCA-transformed data
fig, ax = plt.subplots(figsize=(9, 7))

colors  = ['#e74c3c', '#3498db', '#2ecc71']
markers = ['o', 's', '^']
class_labels = ['Class 1 (Cultivar 1)', 'Class 2 (Cultivar 2)', 'Class 3 (Cultivar 3)']

for cls, color, marker, label in zip([0, 1, 2], colors, markers, class_labels):
    mask = y_wine == cls
    ax.scatter(X_wine_pca[mask, 0], X_wine_pca[mask, 1],
               c=color, marker=marker, s=90, alpha=0.8,
               edgecolors='white', linewidth=0.5, label=label)

ax.set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=12)
ax.set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=12)
ax.set_title('PCA — Wine Dataset: 2D Transformed Feature Space\n(13 features -> 2 principal components)', fontsize=13)
ax.legend(loc='upper right', framealpha=0.9)
ax.axhline(0, color='grey', linewidth=0.5, linestyle='--', alpha=0.5)
ax.axvline(0, color='grey', linewidth=0.5, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

**Interpretation:**  
The 2D PCA scatter plot reveals clear **cluster separation** between the three wine cultivars, especially between Classes 1 and 3. This visual separation in only 2 dimensions — reduced from 13 — demonstrates that PCA successfully preserved the most discriminative structure of the dataset:
- **Class 1** (red circles) clusters at the **positive end of PC1**, indicating high values of the features contributing positively to PC1 (e.g., flavanoids, phenols).
- **Class 3** (green triangles) clusters at the **negative end of PC1**, with lower flavanoid content.
- **Class 2** (blue squares) occupies an intermediate region and shows some overlap with both other classes.

The axes (PC1, PC2) are orthogonal linear combinations of the original 13 features, chosen to maximise variance. They have no direct physical interpretation but encode the dominant patterns of variation in the dataset.

---
## B4. PCA Component Loadings — What Do PC1 & PC2 Represent?

In [ ]:
# Component loadings (eigenvectors)
loadings = pd.DataFrame(
    pca_2d.components_.T,
    index=wine.feature_names,
    columns=['PC1', 'PC2']
).round(4)

print('=== PCA Component Loadings (Feature Contributions) ===')
print(loadings.to_string())

In [ ]:
# Visualise loadings
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for i, (pc, color) in enumerate(zip(['PC1', 'PC2'], ['#3498db', '#e67e22'])):
    sorted_load = loadings[pc].sort_values()
    bar_colors = [color if v >= 0 else '#e74c3c' for v in sorted_load.values]
    axes[i].barh(sorted_load.index, sorted_load.values,
                 color=bar_colors, edgecolor='white', linewidth=0.8)
    axes[i].axvline(0, color='black', linewidth=1)
    axes[i].set_title(f'{pc} Loadings\n({pca_2d.explained_variance_ratio_[i]*100:.1f}% variance explained)')
    axes[i].set_xlabel('Loading Coefficient')

plt.suptitle('PCA Component Loadings — Feature Contributions to PC1 & PC2',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Interpretation — PC1 & PC2 Significance:**

**PC1 (dominant axis):**  
PC1 is primarily driven by phenolic compound features — flavanoids, total phenols, OD280/OD315 of diluted wines, and proanthocyanins have large positive loadings, while colour intensity, malic acid, and hue have negative loadings. PC1 can be interpreted as a **"phenolic richness"** axis: wines scoring high on PC1 are flavanoid-rich, while those scoring low are colour-intense with higher malic acid. Class 1 wines tend to be phenol-rich; Class 3 wines tend to be higher in colour intensity.

**PC2 (secondary axis):**  
PC2 is influenced by different features — alcohol and ash alkalinity contribute positively, while proline and malic acid contribute negatively. PC2 captures a secondary pattern of variation related to **mineralisation and fermentation by-products** rather than phenolic content. It helps separate samples that PC1 cannot distinguish.

---
## B5. Original vs. Transformed Dataset — Comparison

In [ ]:
import time

# Benchmark computational time for a simple classification task
from sklearn.svm import SVC as SVC_bench

# Time on original 13-feature data
t0 = time.perf_counter()
for _ in range(100):
    SVC_bench(kernel='rbf').fit(X_wine_sc, y_wine)
t_orig = (time.perf_counter() - t0) / 100

# Time on PCA-reduced 2-feature data
t0 = time.perf_counter()
for _ in range(100):
    SVC_bench(kernel='rbf').fit(X_wine_pca, y_wine)
t_pca = (time.perf_counter() - t0) / 100

speedup = t_orig / t_pca

comparison = pd.DataFrame({
    'Attribute': [
        'Number of Features',
        'Explained Variance Retained',
        'Avg Training Time (SVM, per run)',
        'Memory (approx.)',
        'Human Interpretable',
        'Visualisable'
    ],
    'Original Dataset': [
        '13',
        '100%',
        f'{t_orig*1000:.2f} ms',
        f'{X_wine_sc.nbytes / 1024:.1f} KB',
        'Yes (named features)',
        'No (13D)'
    ],
    'PCA-Reduced (2 PCs)': [
        '2',
        f'{sum(pca_2d.explained_variance_ratio_)*100:.1f}%',
        f'{t_pca*1000:.2f} ms',
        f'{X_wine_pca.nbytes / 1024:.1f} KB',
        'Partial (abstract axes)',
        'Yes (2D scatter)'
    ]
})

print('=== Original vs. PCA-Transformed Dataset Comparison ===')
print(comparison.to_string(index=False))
print(f'\nSpeedup factor (PCA vs original) : {speedup:.1f}x faster')

**Interpretation:**  

| Aspect | Original (13 features) | PCA (2 PCs) |
|--------|------------------------|-------------|
| **Dimensionality** | 13 features | 2 principal components |
| **Information** | 100% (complete) | ~55% (first 2 PCs) |
| **Computational Efficiency** | Baseline | Significantly faster |
| **Visualisability** | Not directly | Yes (2D scatter) |
| **Interpretability** | Named, physical features | Abstract linear combinations |

**Key trade-off:** Reducing from 13 to 2 components dramatically improves computational efficiency and enables direct visualisation, but retains only ~55% of the total variance. For tasks requiring maximum predictive power, more components (e.g., those capturing >= 95% variance) should be retained. For exploratory analysis and visualisation, 2 components are ideal.

---
## B6. PCA — Advantages, Limitations, and Applications

### Advantages of PCA
1. **Curse of Dimensionality Mitigation:** Reduces the number of features, preventing overfitting in high-dimensional datasets where samples are sparse relative to feature count.
2. **Computational Efficiency:** Fewer features reduce training time and memory usage for downstream models.
3. **Noise Reduction:** Minor principal components (low eigenvalue) often capture noise rather than signal; discarding them can improve model generalisation.
4. **Multicollinearity Elimination:** PCA produces orthogonal (uncorrelated) components, resolving multicollinearity issues that affect regression models.
5. **Visualisation:** Reduces data to 2D or 3D for visual exploration, enabling pattern discovery and cluster identification.

### Limitations of PCA
1. **Information Loss:** Discarding components below the variance threshold discards some information; this may be harmful if the discarded variance is class-discriminative.
2. **Linearity Assumption:** PCA only captures **linear** relationships between features. Nonlinear structures (e.g., manifolds) require Kernel PCA or t-SNE.
3. **Interpretability Loss:** Principal components are abstract linear combinations of original features, making them hard to explain to domain experts.
4. **Unsupervised:** PCA ignores class labels; components maximising variance may not maximise class separability (unlike LDA).
5. **Sensitive to Outliers:** PCA is variance-based; outliers disproportionately inflate variance and can distort principal components.
6. **Scale Sensitivity:** Requires standardisation before application; raw PCA on un-normalised features is misleading.

### Applications of PCA in Machine Learning
| Application Domain | Use Case |
|--------------------|----------|
| Computer Vision | Eigenfaces for face recognition; image compression |
| Natural Language Processing | Latent Semantic Analysis (LSA); word embedding compression |
| Finance | Portfolio risk analysis; factor modelling |
| Genomics | SNP data analysis; population structure discovery |
| Anomaly Detection | Identify outliers as points far from the principal subspace |
| Medical Imaging | fMRI data reduction; EEG signal decomposition |
| Preprocessing | Feature extraction before SVM, k-NN, or neural networks |

---
# Extra Credit: Linear Discriminant Analysis (LDA)
## Aim
To apply LDA to the UCI Wine dataset, reduce it to 2 linear discriminants, and compare the results with PCA in terms of dimensionality reduction and class separability.

---
## EC1. Apply LDA & Visualise

In [ ]:
# LDA: reduce to 2 linear discriminants
# Note: LDA uses class labels (supervised), unlike PCA
lda = LDA(n_components=2)
X_wine_lda = lda.fit_transform(X_wine_sc, y_wine)

print('=== LDA Summary ===')
print(f'Input features     : {X_wine_sc.shape[1]}')
print(f'Output components  : {X_wine_lda.shape[1]}')
print(f'Transformed shape  : {X_wine_lda.shape}')
print(f'\nExplained Variance Ratio (LDA):')
for i, evr in enumerate(lda.explained_variance_ratio_):
    print(f'  LD{i+1}: {evr:.4f} ({evr*100:.2f}%)')
print(f'  Total: {sum(lda.explained_variance_ratio_):.4f} ({sum(lda.explained_variance_ratio_)*100:.2f}%)')

In [ ]:
# Side-by-side comparison: PCA vs LDA scatter plots
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors  = ['#e74c3c', '#3498db', '#2ecc71']
markers = ['o', 's', '^']
class_labels = ['Class 1', 'Class 2', 'Class 3']

datasets = [
    (X_wine_pca, 'PCA', f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)',
     f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)'),
    (X_wine_lda, 'LDA', f'LD1 ({lda.explained_variance_ratio_[0]*100:.1f}%)',
     f'LD2 ({lda.explained_variance_ratio_[1]*100:.1f}%)')
]

for ax, (X_2d, method, xlabel, ylabel) in zip(axes, datasets):
    for cls, color, marker, label in zip([0, 1, 2], colors, markers, class_labels):
        mask = y_wine == cls
        ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
                   c=color, marker=marker, s=90, alpha=0.85,
                   edgecolors='white', linewidth=0.5, label=label)
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(f'{method} — 2D Transformed Feature Space', fontsize=13)
    ax.axhline(0, color='grey', linewidth=0.5, linestyle='--', alpha=0.5)
    ax.axvline(0, color='grey', linewidth=0.5, linestyle='--', alpha=0.5)
    ax.legend(loc='upper right', framealpha=0.9)

plt.suptitle('PCA vs. LDA — Wine Dataset: 2D Dimensionality Reduction Comparison',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## EC2. Quantify Class Separability

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Measure class separability via kNN 5-fold CV accuracy
knn = KNeighborsClassifier(n_neighbors=5)

cv_pca  = cross_val_score(knn, X_wine_pca, y_wine, cv=5, scoring='accuracy')
cv_lda  = cross_val_score(knn, X_wine_lda, y_wine, cv=5, scoring='accuracy')
cv_orig = cross_val_score(knn, X_wine_sc,  y_wine, cv=5, scoring='accuracy')

print('=== Class Separability — 5-Fold CV Accuracy (kNN k=5) ===')
print(f'Original 13 features : {cv_orig.mean():.4f} +/- {cv_orig.std():.4f}')
print(f'PCA (2 components)   : {cv_pca.mean():.4f} +/- {cv_pca.std():.4f}')
print(f'LDA (2 components)   : {cv_lda.mean():.4f} +/- {cv_lda.std():.4f}')

In [ ]:
# Bar chart comparison
fig, ax = plt.subplots(figsize=(8, 5))

methods = ['Original\n(13 features)', 'PCA\n(2 components)', 'LDA\n(2 components)']
means   = [cv_orig.mean(), cv_pca.mean(), cv_lda.mean()]
stds    = [cv_orig.std(),  cv_pca.std(),  cv_lda.std()]
colors  = ['#95a5a6', '#3498db', '#e74c3c']

bars = ax.bar(methods, means, color=colors, edgecolor='white',
              linewidth=1.2, width=0.4, yerr=stds, capsize=6)
for bar, val in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{val:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=11)

ax.set_ylim(0, 1.15)
ax.set_ylabel('5-Fold CV Accuracy (kNN k=5)', fontsize=12)
ax.set_title('Class Separability Comparison: Original vs PCA vs LDA\n(Wine Dataset)', fontsize=13)
ax.axhline(1.0, color='grey', linestyle='--', linewidth=1, alpha=0.5)
plt.tight_layout()
plt.show()

---
## EC3. PCA vs. LDA — Comprehensive Comparison

**Detailed Comparison: PCA vs. LDA**

| Criterion | PCA | LDA |
|-----------|-----|-----|
| **Learning Type** | Unsupervised | Supervised |
| **Objective** | Maximise total variance | Maximise between-class / within-class variance ratio |
| **Uses Class Labels** | No | Yes |
| **Max Components** | min(n_samples, n_features) | n_classes - 1 |
| **For Wine (3 classes)** | Up to 13 components | Up to 2 components |
| **Variance Explained (2D)** | ~55% | ~99%+ (class-discriminative) |
| **Class Separability** | Moderate | High |
| **Interpretability** | Abstract principal axes | Class-discriminant directions |
| **Assumption** | No distribution assumption | Gaussian classes, equal covariance |
| **Outlier Sensitivity** | High | Moderate |
| **Best For** | Exploratory analysis, noise reduction | Classification preprocessing |

**Key Observations:**

1. **Class Separability:** LDA achieves superior class separation compared to PCA in 2D because it explicitly optimises for discriminative directions. In the scatter plot, LDA clusters are visibly tighter and more separated.

2. **Variance Explained:** LDA's explained variance ratio refers to between-class scatter explained, not total dataset variance. LDA's LD1 typically explains >= 68% of the between-class variance for the Wine dataset, while PCA's PC1 only captures total dataset variance.

3. **kNN Accuracy:** LDA-reduced features yield higher 5-fold CV accuracy than PCA-reduced features, confirming better class separability in the 2D LDA projection.

4. **When to Use PCA:** When class labels are unavailable, when the goal is general noise reduction or feature compression, or when inter-class separability is not the primary concern.

5. **When to Use LDA:** When class labels are available and the goal is to find the most discriminative low-dimensional representation for classification tasks.

6. **Maximum Components:** LDA is bounded by `n_classes - 1` components (2 for Wine's 3 classes). For datasets with many classes, PCA retains more flexibility.

---
## EC4. Summary of Observations

**Summary — Extra Credit (LDA vs PCA):**

LDA consistently outperforms PCA as a preprocessing step for classification on the Wine dataset. In the 2D scatter plots, LDA creates clearly separated, compact clusters with minimal overlap between cultivars, whereas PCA's 2D projection shows some overlap between Classes 1 and 2. This is because **LDA is a supervised technique** — it leverages class label information to find linear discriminants that maximise the Fisher criterion (between-class scatter / within-class scatter), making it inherently better suited to classification tasks.

PCA's advantage lies in its unsupervised nature: it can be applied without labels and is a powerful general-purpose tool for dimensionality reduction, noise removal, and data exploration. LDA's limitation is its requirement for class labels and its assumption of Gaussian-distributed, equal-covariance classes — conditions that may not hold in all datasets.

For the Wine dataset specifically:
- **PCA (2D)** retains ~55% variance and shows moderate cluster separation.
- **LDA (2D)** maximises class discriminability and shows excellent separation, especially between Class 1 and Classes 2/3.

**Conclusion:** For classification problems where labelled data is available, LDA is the preferred dimensionality-reduction technique due to its superior class separability. PCA remains the go-to method for exploratory data analysis, unsupervised feature compression, and scenarios with unlabelled data.